# A3 — Cross-Tool Severity Disagreement (Cohen's κ)

How far do the L2 tools disagree about how severe the same control is? Logic lives in `src/misconfig/disagreement.py` (CLAUDE.md rule 9).

**Three constraints shape this experiment, each a finding in its own right:**

1. **Checkov ships no severities** in its open-source repo (0 of 1,684 checks) — they live in the commercial Bridgecrew platform. It therefore cannot take part in a severity agreement statistic at all. Agreement is measured **Prowler vs ScoutSuite only**.
2. **The scales differ**: Prowler rates on five levels, ScoutSuite on two (`danger`/`warning`). κ over mismatched scales would measure the mapping as much as the tools, so the primary statistic collapses both to a shared binary scale (high-risk vs lower). The full 5×2 cross-tab is reported too.
3. **The tools share no identifiers**, so pairs come from semantic matching of check text, constrained to the same provider. The similarity threshold is a free parameter, so κ is reported across a **sweep** of thresholds rather than at one hand-picked value.

A note on what was *not* done: ScoutSuite ships a CIS 1.2.0 ruleset with recommendation numbers and Prowler ships CIS 1.4/1.5/2.0/3.0, but the sections were renumbered between those versions (CIS 1.2.0's §2.1 is §3.1 from 1.4 onward), so joining on recommendation numbers would silently produce wrong pairs.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.misconfig.disagreement import run, log_stats

In [ ]:
results = run()
log_stats(results)

## Threshold sensitivity

κ never leaves the "slight" band across the whole sweep, so the conclusion does not depend on the matcher's threshold.

In [ ]:
results["sweep"]

## Where the tools disagree

Inspect the matched pairs directly — the disagreements are systematic rather than random. CloudWatch alarm checks are the clearest case: Prowler rates every one `medium`, ScoutSuite rates every one `danger`.

In [ ]:
results["by_service"].head(15)

In [ ]:
cols = [
    "similarity",
    "prowler_title",
    "scoutsuite_title",
    "prowler_severity_raw",
    "scoutsuite_severity_raw",
]
results["disagreements"][cols].head(20)

## Match quality check

κ is only meaningful if the pairs really are the same control. Read the highest- and lowest-similarity pairs and judge them by eye before trusting the statistic.

In [ ]:
pairs = results["pairs"]
display(pairs[cols].head(10))
display(pairs[cols].tail(10))